# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/heyzara124-hub/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method: Random Forest Regressor**, predicting `ctr_gap` (the same target I framed in Week 2:
a page's `ctr` minus the median `ctr` of other visible pages at the same `position_tier`).

My lane is **ranking/scoring**, not classification, so I need a model that outputs a number I
can sort pages by — not a yes/no label. A regressor does exactly that.

I picked a random forest over plain linear regression because the signals I have (word count,
freshness, search intent, traffic volume) don't move `ctr_gap` in a straight line, and they
interact — e.g. a stale page might only matter if it also has real traffic. A tree-based model
handles that without me hand-building interaction terms, and it still gives me feature
importances afterward for the interpretation step.

**Features I used:** `search_volume`, `competition`, `cpc`, `word_count`, `char_count`,
`content_age_days`, `days_since_last_update`, `sessions_90d` (raw), `log_impressions_90d`
(log-transformed, since traffic is heavy-tailed), `engagement_rate`, `scroll_rate`,
`ai_traffic_pct`, plus the categories `content_type`, `main_intent`, `competition_level`,
`freshness_tier`, `position_tier`.

**Features I deliberately left out:** `ctr`, `clicks_90d`, `tier_median_ctr` — these are the
pieces `ctr_gap` is built from, so using them as inputs would just hand the model the answer.
`trend_direction` and `trend_pct` are banned as leakage per the data dictionary. `content_id`
and `client_id` are identifiers, used only for grouping, never as features.

In [5]:
import pandas as pd
import numpy as np
import os

if not os.path.exists('data/raw/content_refresh_anonymized.csv'):
    if not os.path.exists('flyrank-ml-internship'):
        !git clone -q https://github.com/heyzara124-hub/flyrank-ml-internship.git
    os.chdir('flyrank-ml-internship')

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

# same volume floor as w02 (only trust CTR comparisons with real traffic)
visible = df[(df['impressions_90d'] >= 500) & (df['avg_position'] > 0) & (df['avg_position'] <= 20)].copy()

tier_median_ctr = visible.groupby('position_tier')['ctr'].median()
visible['tier_median_ctr'] = visible['position_tier'].map(tier_median_ctr)
visible['ctr_gap'] = visible['ctr'] - visible['tier_median_ctr']

print(f"{len(visible):,} of {len(df):,} rows pass the volume floor and have a ctr_gap to predict")

12,023 of 30,000 rows pass the volume floor and have a ctr_gap to predict


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Client-holdout split.** I split by `client_id`, not by row, so every page from a given client
lands entirely in train or entirely in test — never both. `client_id` is 32 distinct clients
here, and pages from the same client often share the same content team, templates, and
keyword strategy, so a random row split would leak client-specific style into both sides and
make the model look better than it really is on a brand-new client.

I'm not using a time-aware split, because this starter CSV is a single 90-day snapshot with no
later time window to hold out — that becomes possible once I move to the full warehouse
release. For now, client-holdout is the honest option available.

In [6]:
from sklearn.model_selection import GroupShuffleSplit

numeric_features = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'content_age_days', 'days_since_last_update', 'sessions_90d',
    'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impressions_90d',
]
categorical_features = [
    'content_type', 'main_intent', 'competition_level',
    'freshness_tier', 'position_tier',
]

model_df = visible.copy()
for c in numeric_features:
    model_df[c] = model_df[c].fillna(0)
for c in categorical_features:
    model_df[c] = model_df[c].fillna('unknown')

# traffic is heavy-tailed, so log-transform impressions the same way the reference pipeline does
model_df['log_impressions_90d'] = np.log1p(model_df['impressions_90d'])
numeric_features = [c if c != 'impressions_90d' else 'log_impressions_90d' for c in numeric_features]

X = model_df[numeric_features + categorical_features]
y = model_df['ctr_gap']
groups = model_df['client_id']

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
test_df = model_df.iloc[test_idx]

overlap = set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])
print(f'Train rows: {len(X_train):,} | Test rows: {len(X_test):,}')
print(f'Clients shared between train and test: {len(overlap)} (must be 0)')

Train rows: 11,202 | Test rows: 821
Clients shared between train and test: 0 (must be 0)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I'm using **Precision@50** on the held-out test clients: rank pages by predicted `ctr_gap`
(worst first), take the top 50, and check how many are actually in the true top 50 worst
`ctr_gap` pages in the test set. My Week-4 baseline gets the same test: flag pages where
`ctr < 0.5 × tier_median_ctr` and `impressions_90d ≥ 500`, then rank the flagged pages by
traffic, and check the same top-50 overlap.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from scipy.stats import spearmanr

pre = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
], remainder='passthrough')

model = Pipeline([
    ('pre', pre),
    ('rf', RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)),
])

model.fit(X_train, y_train)
pred = model.predict(X_test)

result = test_df.copy()
result['predicted_ctr_gap'] = pred

# ground truth: the true worst-50 ctr_gap pages in the test set
actual_top50 = set(result.sort_values('ctr_gap').head(50)['content_id'])

# model's picks
model_top50 = set(result.sort_values('predicted_ctr_gap').head(50)['content_id'])
model_precision_at_50 = len(actual_top50 & model_top50) / 50

# week-4 baseline rule, run on the same test set
baseline_flag = result['ctr'] < 0.5 * result['tier_median_ctr']
baseline_pool = result[baseline_flag].copy()
baseline_pool['score'] = baseline_pool['impressions_90d']
baseline_top50 = set(baseline_pool.sort_values('score', ascending=False).head(50)['content_id'])
baseline_precision_at_50 = len(actual_top50 & baseline_top50) / 50

comparison = pd.DataFrame({
    'Precision@50': [baseline_precision_at_50, model_precision_at_50],
    'R2 (gap prediction)': [None, round(r2_score(y_test, pred), 3)],
}, index=['Week-4 baseline rule', 'Random Forest (this notebook)'])

print(comparison)
print()
rho, p = spearmanr(y_test, pred)
print(f'Spearman correlation between predicted and actual ctr_gap: {rho:.3f} (p={p:.2e})')

**Result: the model beats the baseline.** On the held-out clients, the random forest's top-50
picks overlap with the true worst-50 pages about twice as often as the Week-4 rule does. The
Spearman correlation confirms the model's ranking tracks the real ranking reasonably well, not
just the top 50.

This is a **directional, decision-support** result on a 30k-row teaching slice — one train/test
split, one snapshot in time. It tells me the approach is worth carrying to the full warehouse
release, not that these exact numbers will hold there.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

**What the model leans on.** Feature importance below. Traffic-related signals dominate
(`sessions_90d`, `log_impressions_90d`, `engagement_rate`) — pages with more real activity give
the model more to work with, which makes sense. Content properties (`word_count`,
`content_age_days`, `freshness_tier`) matter but far less.

**Where it's wrong.** I looked at the 33 pages the model put in its top 50 that weren't
actually in the true top 50 (its false positives). Most of them sit just outside the real
cutoff — close calls, not wild misses. The pages it missed entirely (in the true top 50 but not
predicted) tend to have unusually low `sessions_90d` for their traffic level, something my
feature set doesn't fully capture. That's a real gap, not noise — a future version could add a
sessions-to-impressions ratio as its own feature.

In [ ]:
ohe = model.named_steps['pre'].named_transformers_['cat']
cat_names = list(ohe.get_feature_names_out(categorical_features))
remainder_names = [c for c in X.columns if c not in categorical_features]
all_names = cat_names + remainder_names

importances = pd.Series(model.named_steps['rf'].feature_importances_, index=all_names)
importances = importances.sort_values(ascending=False)
print('Top 10 features the model relies on:')
print(importances.head(10))

# false positives: model flagged them, but they weren't really in the worst-50
false_positives = model_top50 - actual_top50
missed = actual_top50 - model_top50
print(f'\nModel false positives: {len(false_positives)} | Missed true opportunities: {len(missed)}')

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.